Scaled dot-product attention from scratch

The core attention mechanism with optional masking

In [6]:
import torch
import torch.nn.functional as F
import math

# scaled dot product attention

def scaled_dot_product_attention(Q, K, V, mask=None):
  """
    Computes attention(Q, K, V) = softmax(QK^T / sqrt(d_k)) * V

    Args:
        Q: Queries  (batch, seq_len, d_k)
        K: Keys     (batch, seq_len, d_k)
        V: Values   (batch, seq_len, d_v)
        mask: Optional boolean mask (True = attend, False = block)

    Returns:
        output:  (batch, seq_len, d_v)
        weights: (batch, seq_len, seq_len)
  """
  d_k = Q.size(-1)

  # step 1: compute raw attention scores
  scores = torch.matmul(Q, K.transpose(-2,-1)) #  (batch,n, n)

  # step 2: scale by sqrt(d_k) to prevent gradient issues
  scores = scores  / math.sqrt(d_k)

  # step 3: Apply mask (eg causal mask for decoders)
  if mask is not None:
    scores = scores.masked_fill(~mask, float('-inf'))

  # step 4: softmax -> each row sums to 1
  weights = F.softmax(scores, dim=-1)

  # step 5: weighted sum of values
  output = torch.matmul(weights, V)

  return output, weights

batch_size, seq_len, d_model = 1, 6, 64

# simulate token embeddings
X = torch.randn(batch_size, seq_len, d_model)

# Learned projection matrices
W_q = torch.randn(d_model, d_model)
W_k = torch.randn(d_model, d_model)
W_v = torch.randn(d_model, d_model)

# Project input into Q, K, V
Q = X @ W_q  # (1, 6, 64)
K = X @ W_k  # (1, 6, 64)
V = X @ W_v  # (1, 6, 64)

# compute attention bidirectional - no mask
output, weights = scaled_dot_product_attention(Q, K, V)

print(f"Input shape:      {X.shape}")       # [1, 6, 64]
print(f"Output shape:     {output.shape}")   # [1, 6, 64]
print(f"Attention matrix: {weights.shape}")  # [1, 6, 6]
print(f"Row sums to 1:    {weights[0, 0].sum():.4f}")

# causal mask for gpt style
causal_mask = torch.tril(torch.ones(seq_len, seq_len)).bool()
# [[True, False, False, ...],
#  [True, True,  False, ...],
#  [True, True,  True,  ...]]

output_causal, weights_causal = scaled_dot_product_attention(
    Q, K, V, mask=causal_mask
)

print(f"\nCausal weights (row 0): {weights_causal[0, 0]}")
# Only first position has non-zero weight


Input shape:      torch.Size([1, 6, 64])
Output shape:     torch.Size([1, 6, 64])
Attention matrix: torch.Size([1, 6, 6])
Row sums to 1:    1.0000

Causal weights (row 0): tensor([1., 0., 0., 0., 0., 0.])


Visualizing attention patterns

See what words attend to and compare scaled vs unscaled




In [8]:
import torch
import torch.nn.functional as F
import math

# attention visualization for a sentence
sentence = ["The", "cat", "sat", "on", "the", "mat"]
n = len(sentence)
d_k = 32

# Simulated Q, K after projection (in practice, learned)
torch.manual_seed(42)
Q = torch.randn(n, d_k)
K = torch.randn(n, d_k)

# compute scaled attention weights
scores = torch.matmul(Q, K.T) / math.sqrt(d_k)
weights = F.softmax(scores, dim = -1)

# print attention  heatmap
print(f'Attention weights (from -> to): \n')
header = "        " + "  ".join(f"{w:>5}" for w in sentence)
print("        " + "-" * 42)

for i, word in enumerate(sentence):
  row = "  ".join(f"{weights[i, j]:.2f}" for j in range(n))
  print(f"{word:>6}: {row}")

# find strongest attention for each word
print("\nStrongest attention per word:")
for i, word in enumerate(sentence):
  top_j = weights[i].argmax().item()
  strength = weights[i, top_j].item()
  print(f"  '{word}' -> '{sentence[top_j]}' ({strength:.2f})")

# compare with vs without scaling
print("\n--- Effect of Scaling (d=512) ---")
d_large = 512
Q_large = torch.randn(n, d_large)
K_large = torch.randn(n, d_large)

# Without scaling (BROKEN - gradients vanish)
scores_raw = Q_large @ K_large.T
weights_raw = F.softmax(scores_raw, dim=-1)

# With scaling (CORRECT - gradients healthy)
scores_scaled = (Q_large @ K_large.T) / math.sqrt(d_large)
weights_scaled = F.softmax(scores_scaled, dim=-1)

print(f"Max score without scaling: {scores_raw.max():.1f}")
print(f"Max score with scaling:    {scores_scaled.max():.1f}")
print(f"Max weight without scaling: {weights_raw.max():.4f} (near 1.0!)")
print(f"Max weight with scaling:    {weights_scaled.max():.4f} (distributed)")

Attention weights (from -> to): 

        ------------------------------------------
   The: 0.09  0.19  0.18  0.01  0.40  0.13
   cat: 0.38  0.10  0.16  0.05  0.12  0.19
   sat: 0.36  0.13  0.24  0.07  0.10  0.10
    on: 0.15  0.20  0.37  0.04  0.14  0.10
   the: 0.16  0.17  0.06  0.04  0.19  0.37
   mat: 0.28  0.07  0.08  0.20  0.06  0.31

Strongest attention per word:
  'The' -> 'the' (0.40)
  'cat' -> 'The' (0.38)
  'sat' -> 'The' (0.36)
  'on' -> 'sat' (0.37)
  'the' -> 'mat' (0.37)
  'mat' -> 'mat' (0.31)

--- Effect of Scaling (d=512) ---
Max score without scaling: 46.1
Max score with scaling:    2.0
Max weight without scaling: 1.0000 (near 1.0!)
Max weight with scaling:    0.6974 (distributed)
